# Probability for Artificial Intelligence

The real world is uncertain. Sensors are noisy, data is incomplete, users behave unpredictably, and models make imperfect predictions. **Probability** is the mathematical framework for reasoning under uncertainty — quantifying how likely events are, updating beliefs when new evidence arrives, and making decisions when outcomes are not guaranteed.

In artificial intelligence, probability appears everywhere: spam filters estimate the probability that an email is junk, medical AI reports the probability of disease, language models assign probabilities to the next word, generative models learn probability distributions over images and text, and reinforcement learning agents maximize expected reward. This notebook builds probability from foundational concepts through distributions, Bayes' theorem, entropy, and practical AI applications — with formulas explained clearly and demonstrated in Python.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Probability Matters in AI

Early AI pursued logical certainty — rules that were true or false. Real-world AI must handle **uncertainty**. A face recognition system does not say "this is definitely Alice"; it says "there is a 97% probability this face matches Alice." A weather model does not predict with certainty; it predicts a 70% chance of rain.

Probability enables AI systems to:

- **Quantify confidence** in predictions rather than giving false certainty.
- **Update beliefs** when new data arrives (Bayesian inference).
- **Make optimal decisions** under uncertainty (expected utility, risk).
- **Learn distributions** from data (generative models, maximum likelihood).
- **Measure information** and surprise (entropy, cross-entropy as loss functions).

Modern machine learning is deeply probabilistic even when not explicitly labeled as such. Training a classifier with cross-entropy loss is fitting a probability model. Sampling from a diffusion model is drawing from a learned probability distribution. Understanding probability is understanding what these systems actually compute.

---

## 2. Fundamental Concepts

### 2.1 Sample Space and Events

An **experiment** is any process with uncertain outcomes. The **sample space** $\Omega$ is the set of all possible outcomes. An **event** is a subset of the sample space — a collection of outcomes we care about.

Example: Rolling a die.
- Sample space: $\Omega = \{1, 2, 3, 4, 5, 6\}$
- Event "rolling an even number": $E = \{2, 4, 6\}$
- Event "rolling greater than 4": $F = \{5, 6\}$

### 2.2 Probability Axioms (Kolmogorov)

Probability assigns a number $P(A)$ to each event $A$, satisfying three axioms:

1. **Non-negativity:** $P(A) \geq 0$ for any event $A$.
2. **Normalization:** $P(\Omega) = 1$ — something must happen.
3. **Additivity:** For mutually exclusive events $A$ and $B$: $P(A \cup B) = P(A) + P(B)$.

From these axioms, it follows that $P(A) \in [0, 1]$, $P(\emptyset) = 0$, and $P(\neg A) = 1 - P(A)$.

### 2.3 Classical and Empirical Probability

**Classical (theoretical):** If all outcomes are equally likely, $P(A) = \frac{\text{number of favorable outcomes}}{\text{total outcomes}}$. For a fair die, $P(\text{even}) = \frac{3}{6} = 0.5$.

**Empirical (frequentist):** Probability is the long-run relative frequency of an event. Flip a coin 10,000 times; the fraction of heads approaches 0.5. This connects probability to data — the foundation of statistical machine learning.

In [ ]:
# Empirical probability: simulate coin flips
n_flips = 10_000
flips = np.random.choice(["H", "T"], size=n_flips)
p_heads = (flips == "H").mean()

print(f"After {n_flips} fair coin flips:")
print(f"  P(Heads) ≈ {p_heads:.4f}  (theoretical: 0.5000)")

# Running proportion converges to 0.5
running = np.cumsum(flips == "H") / np.arange(1, n_flips + 1)
plt.figure(figsize=(9, 4))
plt.plot(running, linewidth=1)
plt.axhline(0.5, color="r", linestyle="--", label="True probability = 0.5")
plt.xlabel("Number of flips")
plt.ylabel("Proportion of heads")
plt.title("Law of Large Numbers: Empirical Probability Converges")
plt.legend()
plt.show()

---

## 3. Conditional Probability

Often we want the probability of event $A$ **given that** event $B$ has already occurred. This is **conditional probability**:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)} \qquad \text{provided } P(B) > 0$$

Read as "probability of $A$ given $B$." We restrict attention to outcomes where $B$ is true, then ask what fraction of those also satisfy $A$.

**Example:** In a deck of 52 cards, what is the probability of drawing a King given that the card is a face card? There are 12 face cards and 4 Kings among them:

$$P(\text{King} \mid \text{Face}) = \frac{4}{12} = \frac{1}{3}$$

Conditional probability is central to AI: $P(\text{disease} \mid \text{symptoms})$, $P(\text{spam} \mid \text{email text})$, $P(\text{next word} \mid \text{previous words})$.

In [ ]:
# Medical test example (conditional probability)
# Disease prevalence: 1% of population
# Test sensitivity: P(positive | disease) = 99%
# Test specificity: P(negative | no disease) = 95%

P_disease = 0.01
P_pos_given_disease = 0.99
P_neg_given_no_disease = 0.95
P_pos_given_no_disease = 1 - P_neg_given_no_disease

P_pos = P_pos_given_disease * P_disease + P_pos_given_no_disease * (1 - P_disease)
P_disease_given_pos = (P_pos_given_disease * P_disease) / P_pos

print("Medical test scenario:")
print(f"  P(disease) = {P_disease:.2%}")
print(f"  P(positive | disease) = {P_pos_given_disease:.2%}")
print(f"  P(negative | no disease) = {P_neg_given_no_disease:.2%}")
print(f"\n  P(positive test) = {P_pos:.4f}")
print(f"  P(disease | positive test) = {P_disease_given_pos:.2%}")
print("\n  Despite 99% sensitivity, only ~17% of positive tests are true positives!")
print("  Low base rate (prevalence) dominates. This is why Bayes' theorem matters.")

---

## 4. Bayes' Theorem

Bayes' theorem reverses conditional probability — it lets us compute $P(A \mid B)$ from $P(B \mid A)$:

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

In AI and statistics, this is usually written as:

$$P(\text{hypothesis} \mid \text{data}) = \frac{P(\text{data} \mid \text{hypothesis}) \cdot P(\text{hypothesis})}{P(\text{data})}$$

The components have names:
- **$P(\text{hypothesis} \mid \text{data})$** — **posterior**: updated belief after seeing data.
- **$P(\text{data} \mid \text{hypothesis})$** — **likelihood**: how probable is the data if the hypothesis is true.
- **$P(\text{hypothesis})$** — **prior**: belief before seeing data.
- **$P(\text{data})$** — **evidence** (marginal likelihood): normalizing constant.

Bayes' theorem is the foundation of **Bayesian machine learning**, **spam filtering** (Naive Bayes), **medical diagnosis**, and any system that updates beliefs from evidence. It formalizes the intuition: combine what you already believed with how well the evidence fits each possibility.

In [ ]:
def bayes(prior, likelihood, evidence):
    """Compute posterior P(hypothesis | data) = likelihood * prior / evidence."""
    return (likelihood * prior) / evidence

# Spam filter: P(spam | contains word "free")
P_spam = 0.30                          # prior: 30% of emails are spam
P_free_given_spam = 0.80               # likelihood: 80% of spam has "free"
P_free_given_ham = 0.05                # 5% of legitimate email has "free"
P_free = P_free_given_spam * P_spam + P_free_given_ham * (1 - P_spam)

P_spam_given_free = bayes(P_spam, P_free_given_spam, P_free)

print("Bayes' theorem — spam filter:")
print(f"  Prior P(spam) = {P_spam:.0%}")
print(f"  P('free' | spam) = {P_free_given_spam:.0%}")
print(f"  P('free' | ham)  = {P_free_given_ham:.0%}")
print(f"  P('free') = {P_free:.4f}")
print(f"\n  Posterior P(spam | 'free') = {P_spam_given_free:.2%}")

---

## 5. Independence

Two events $A$ and $B$ are **independent** if knowing that $B$ occurred does not change the probability of $A$:

$$P(A \mid B) = P(A) \qquad \text{equivalently} \qquad P(A \cap B) = P(A) \cdot P(B)$$

Independent events have no influence on each other. Fair coin flips are independent — previous flips do not affect the next. **Naive Bayes** classifiers assume features are independent given the class label — an approximation that is often wrong but works surprisingly well for text classification.

Events that are not independent are **dependent**. Drawing cards without replacement: $P(\text{second Ace} \mid \text{first was Ace})$ changes because the deck composition changed.

---

## 6. Random Variables

A **random variable** $X$ is a function that assigns a numerical value to each outcome in the sample space. Instead of reasoning about abstract events, we work with numbers.

**Discrete random variables** take countable values (0, 1, 2, ...). Examples: number of heads in 10 coin flips, number of emails received per hour.

**Continuous random variables** take any value in an interval. Examples: height, temperature, model prediction score.

For a discrete random variable, the **probability mass function (PMF)** gives $P(X = x)$ for each possible value $x$. For a continuous random variable, the **probability density function (PDF)** $f(x)$ satisfies $P(a \leq X \leq b) = \int_a^b f(x)\, dx$. Continuous probabilities for exact values are zero — we measure intervals.

---

## 7. Important Discrete Distributions

### 7.1 Bernoulli Distribution

Models a single trial with two outcomes (success/failure). $X \in \{0, 1\}$:

$$P(X = 1) = p, \quad P(X = 0) = 1 - p$$

A single coin flip, a single binary classification (spam or not spam).

### 7.2 Binomial Distribution

Models the number of successes in $n$ independent Bernoulli trials:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

where $\binom{n}{k} = \frac{n!}{k!(n-k)!}$. Example: number of heads in 10 coin flips with $p = 0.5$.

### 7.3 Poisson Distribution

Models the count of events in a fixed interval when events occur independently at a constant average rate $\lambda$:

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

Examples: number of emails per hour, number of typos per page, number of customers arriving per minute.

In [ ]:
k = np.arange(0, 16)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bernoulli
axes[0].bar([0, 1], [0.3, 0.7], color=["steelblue", "coral"], tick_label=["0", "1"])
axes[0].set_title("Bernoulli (p=0.7)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("P(X=x)")

# Binomial
binom_pmf = stats.binom.pmf(k, n=15, p=0.4)
axes[1].bar(k, binom_pmf, color="steelblue")
axes[1].set_title("Binomial (n=15, p=0.4)")
axes[1].set_xlabel("k successes")

# Poisson
poisson_pmf = stats.poisson.pmf(k, mu=4)
axes[2].bar(k, poisson_pmf, color="steelblue")
axes[2].set_title("Poisson (λ=4)")
axes[2].set_xlabel("k events")

plt.tight_layout()
plt.show()

---

## 8. Important Continuous Distributions

### 8.1 Uniform Distribution

Every value in $[a, b]$ is equally likely:

$$f(x) = \frac{1}{b - a} \quad \text{for } x \in [a, b]$$

Used for random initialization of neural network weights and as a simple prior.

### 8.2 Gaussian (Normal) Distribution

The most important continuous distribution in AI and statistics:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Parameters: **mean** $\mu$ (center) and **variance** $\sigma^2$ (spread). Written $X \sim \mathcal{N}(\mu, \sigma^2)$.

The Gaussian appears everywhere: measurement noise, weight initialization, latent spaces in variational autoencoders, diffusion models, and the **Central Limit Theorem** — the sum of many independent random variables tends toward a Gaussian regardless of the original distribution. This theorem explains why Gaussian assumptions work so often in practice.

In [ ]:
x = np.linspace(-5, 5, 300)

plt.figure(figsize=(9, 4))
for mu, sigma in [(0, 1), (0, 2), (2, 0.5)]:
    plt.plot(x, stats.norm.pdf(x, mu, sigma), label=f"μ={mu}, σ={sigma}")

plt.xlabel("x")
plt.ylabel("Probability density")
plt.title("Gaussian (Normal) Distributions")
plt.legend()
plt.show()

# Central Limit Theorem demo: sum of uniform random variables → Gaussian
n_samples = 10_000
uniform_sum = sum(np.random.uniform(0, 1, n_samples) for _ in range(12))

plt.figure(figsize=(8, 4))
plt.hist(uniform_sum, bins=50, density=True, alpha=0.7, label="Sum of 12 uniforms")
mu, sigma = uniform_sum.mean(), uniform_sum.std()
plt.plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=2, label="Fitted Gaussian")
plt.title("Central Limit Theorem")
plt.xlabel("Value")
plt.legend()
plt.show()

---

## 9. Expectation and Variance

The **expected value** (mean) of a random variable $X$ is the long-run average:

**Discrete:** $\mathbb{E}[X] = \sum_x x \cdot P(X = x)$

**Continuous:** $\mathbb{E}[X] = \int_{-\infty}^{\infty} x \cdot f(x)\, dx$

The **variance** measures spread around the mean:

$$\text{Var}(X) = \mathbb{E}[(X - \mu)^2] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

The **standard deviation** $\sigma = \sqrt{\text{Var}(X)}$ is in the same units as $X$.

**Properties:**
- $\mathbb{E}[aX + b] = a\mathbb{E}[X] + b$
- $\text{Var}(aX + b) = a^2 \text{Var}(X)$
- For independent $X, Y$: $\mathbb{E}[X + Y] = \mathbb{E}[X] + \mathbb{E}[Y]$, $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$

In AI, expected value underlies **expected loss**, **expected reward** in reinforcement learning, and **Monte Carlo estimation**.

In [ ]:
# Simulate and compare theoretical vs empirical expectation/variance
samples = np.random.binomial(n=20, p=0.3, size=100_000)

theoretical_mean = 20 * 0.3
theoretical_var = 20 * 0.3 * 0.7

print("Binomial(n=20, p=0.3):")
print(f"  Theoretical E[X] = {theoretical_mean:.2f},  Empirical = {samples.mean():.4f}")
print(f"  Theoretical Var(X) = {theoretical_var:.2f}, Empirical = {samples.var():.4f}")

---

## 10. Joint, Marginal, and Conditional Distributions

When multiple random variables interact, we need **joint distributions**. For two discrete variables $X$ and $Y$, the joint PMF is $P(X = x, Y = y)$.

The **marginal distribution** sums (or integrates) over the other variable:

$$P(X = x) = \sum_y P(X = x, Y = y)$$

The **conditional distribution** fixes one variable and considers the other:

$$P(X = x \mid Y = y) = \frac{P(X = x, Y = y)}{P(Y = y)}$$

In machine learning, high-dimensional data is modeled as joint distributions. A generative model of images learns $P(\text{image})$. A language model learns $P(\text{word}_1, \text{word}_2, \ldots, \text{word}_n)$. Training often factorizes this joint using the chain rule of probability.

In [ ]:
# Joint distribution: weather vs umbrella
# Rows: Rain/No Rain, Cols: Umbrella/No Umbrella
joint = np.array([
    [0.30, 0.10],   # Rain + umbrella, Rain + no umbrella
    [0.05, 0.55],   # No rain + umbrella, No rain + no umbrella
])

print("Joint P(Weather, Umbrella):")
print("              Umbrella  No Umbrella")
print(f"  Rain         {joint[0,0]:.2f}      {joint[0,1]:.2f}")
print(f"  No Rain      {joint[1,0]:.2f}      {joint[1,1]:.2f}")
print(f"  Sum = {joint.sum():.2f}")

P_rain = joint[0].sum()
P_umbrella_given_rain = joint[0, 0] / P_rain
print(f"\n  P(Rain) = {P_rain:.2f}")
print(f"  P(Umbrella | Rain) = {P_umbrella_given_rain:.2f}")

---

## 11. Maximum Likelihood Estimation (MLE)

Given observed data, how do we find the best parameters for a probability distribution? **Maximum Likelihood Estimation** chooses parameters that maximize the probability of observing the data:

$$\hat{\theta} = \arg\max_{\theta} P(\text{data} \mid \theta) = \arg\max_{\theta} \mathcal{L}(\theta)$$

$\mathcal{L}(\theta)$ is the **likelihood function**. In practice, we maximize the **log-likelihood** (sums instead of products, numerically stable):

$$\hat{\theta} = \arg\max_{\theta} \sum_{i} \log P(x_i \mid \theta)$$

Training neural networks with cross-entropy loss is equivalent to maximum likelihood estimation for a categorical distribution. Fitting a Gaussian to data by computing the sample mean and variance is MLE. MLE is the bridge between probability theory and machine learning optimization.

In [ ]:
# MLE for Gaussian: mu_hat = sample mean, sigma_hat = sample std
data = np.random.normal(loc=5.0, scale=2.0, size=500)

mu_mle = data.mean()
sigma_mle = data.std()

print(f"True parameters:     μ=5.0, σ=2.0")
print(f"MLE from {len(data)} samples: μ={mu_mle:.4f}, σ={sigma_mle:.4f}")

x = np.linspace(data.min() - 1, data.max() + 1, 200)
plt.figure(figsize=(8, 4))
plt.hist(data, bins=30, density=True, alpha=0.6, label="Data")
plt.plot(x, stats.norm.pdf(x, mu_mle, sigma_mle), "r-", linewidth=2, label="MLE Gaussian fit")
plt.title("Maximum Likelihood Estimation of Gaussian Parameters")
plt.legend()
plt.show()

---

## 12. Entropy and Information Theory

**Entropy** measures the uncertainty (or average surprise) in a probability distribution. For discrete $X$:

$$H(X) = -\sum_x P(x) \log P(x)$$

High entropy means outcomes are unpredictable (uniform distribution has maximum entropy). Low entropy means outcomes are predictable (a distribution concentrated on one value has entropy near zero).

**Cross-entropy** between true distribution $p$ and predicted distribution $q$:

$$H(p, q) = -\sum_x p(x) \log q(x)$$

In classification, $p$ is the true label (one-hot) and $q$ is the model's predicted probability. **Minimizing cross-entropy loss is equivalent to minimizing the surprise** the model assigns to correct answers. This is why cross-entropy is the standard loss for classification and language modeling.

**KL Divergence** measures how one distribution differs from another:

$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)} = H(p, q) - H(p)$$

In [ ]:
def entropy(probs):
    probs = np.array(probs)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def cross_entropy(p, q):
    p, q = np.array(p), np.array(q)
    mask = p > 0
    return -np.sum(p[mask] * np.log(q[mask]))

print("Entropy (bits):")
print(f"  Fair coin [0.5, 0.5]:     H = {entropy([0.5, 0.5]):.4f}  (max for 2 outcomes)")
print(f"  Certain [1.0, 0.0]:       H = {entropy([1.0, 0.0]):.4f}  (no surprise)")
print(f"  Skewed [0.9, 0.1]:        H = {entropy([0.9, 0.1]):.4f}")

p_true = [0.0, 1.0, 0.0]       # true class index 1
q_good = [0.05, 0.90, 0.05]    # model confident and correct
q_bad = [0.80, 0.10, 0.10]     # model confident but wrong

print(f"\nCross-entropy (3-class):")
print(f"  Good prediction: {cross_entropy(p_true, q_good):.4f}")
print(f"  Bad prediction:  {cross_entropy(p_true, q_bad):.4f}")

---

## 13. Naive Bayes Classifier

The **Naive Bayes** classifier applies Bayes' theorem with a simplifying assumption: features are **conditionally independent** given the class. For class $C$ and features $x_1, x_2, \ldots, x_n$:

$$P(C \mid x_1, \ldots, x_n) \propto P(C) \prod_{i=1}^{n} P(x_i \mid C)$$

Despite the "naive" independence assumption (almost always false in reality), Naive Bayes works well for text classification, spam filtering, and sentiment analysis because it is fast, needs little training data, and the ranking of classes is often correct even when probability estimates are miscalibrated.

In [ ]:
# Tiny Naive Bayes spam classifier with word probabilities
word_probs = {
    "free":   {"spam": 0.70, "ham": 0.05},
    "meeting": {"spam": 0.02, "ham": 0.40},
    "win":    {"spam": 0.50, "ham": 0.01},
    "report": {"spam": 0.03, "ham": 0.35},
}
class_prior = {"spam": 0.30, "ham": 0.70}

def naive_bayes_score(words, label):
    log_prob = np.log(class_prior[label])
    for word in words:
        if word in word_probs:
            log_prob += np.log(word_probs[word][label])
    return log_prob

def classify(email_words):
    scores = {label: naive_bayes_score(email_words, label) for label in ["spam", "ham"]}
    return max(scores, key=scores.get), scores

emails = [
    ["free", "win"],
    ["meeting", "report"],
    ["free", "meeting"],
]

for words in emails:
    label, scores = classify(words)
    print(f"Words {words} → {label}  (log scores: spam={scores['spam']:.2f}, ham={scores['ham']:.2f})")

---

## 14. Monte Carlo Methods

When exact probability calculations are intractable, **Monte Carlo methods** use random sampling to approximate answers. The core idea: simulate the random process many times and use the fraction of outcomes matching the event as the probability estimate.

Monte Carlo appears in AI for:
- **Estimating expectations** (reinforcement learning policy evaluation).
- **Bayesian inference** (Markov Chain Monte Carlo — MCMC).
- **Integration** (estimating areas under complex functions).
- **Game playing** (Monte Carlo Tree Search in AlphaGo).

The **Law of Large Numbers** guarantees that as sample size increases, the Monte Carlo estimate converges to the true value.

In [ ]:
# Monte Carlo: estimate π by randomly dropping points in a square
n_points = 50_000
x = np.random.uniform(-1, 1, n_points)
y = np.random.uniform(-1, 1, n_points)
inside_circle = (x**2 + y**2) <= 1
pi_estimate = 4 * inside_circle.mean()

print(f"Monte Carlo estimate of π with {n_points} points: {pi_estimate:.6f}")
print(f"True π: {np.pi:.6f}")

plt.figure(figsize=(5, 5))
plt.scatter(x[inside_circle], y[inside_circle], s=1, alpha=0.3, label="Inside")
plt.scatter(x[~inside_circle], y[~inside_circle], s=1, alpha=0.3, label="Outside")
plt.axis("equal")
plt.title(f"Monte Carlo π ≈ {pi_estimate:.4f}")
plt.legend(markerscale=5)
plt.show()

---

## 15. Probability in Modern AI

Probability theory connects directly to contemporary AI systems:

**Language models** assign probabilities to sequences: $P(w_1, w_2, \ldots, w_n) = \prod_i P(w_i \mid w_1, \ldots, w_{i-1})$. Training maximizes the log-probability of training text.

**Generative models** (GANs, VAEs, diffusion models) learn to sample from $P(\text{data})$ — the probability distribution of real images, audio, or text.

**Bayesian neural networks** maintain probability distributions over weights rather than point estimates, quantifying uncertainty in predictions.

**Reinforcement learning** maximizes expected cumulative reward: $\mathbb{E}[\sum_t \gamma^t r_t]$.

**Classification with softmax** outputs a probability distribution over classes. The model's confidence is explicit.

**Temperature scaling** in language model sampling adjusts the "sharpness" of the output distribution — higher temperature means more randomness, lower means more deterministic.

In [ ]:
# Temperature scaling: logits → softmax with different temperatures
def softmax_with_temperature(logits, temperature):
    scaled = logits / temperature
    exp = np.exp(scaled - np.max(scaled))
    return exp / exp.sum()

logits = np.array([2.0, 1.0, 0.5, 0.1])
labels = ["A", "B", "C", "D"]

print(f"{'Temp':>6}  " + "  ".join(f"P({l})" for l in labels))
for temp in [0.5, 1.0, 2.0, 5.0]:
    probs = softmax_with_temperature(logits, temp)
    print(f"{temp:>6}  " + "  ".join(f"{p:.3f}" for p in probs))

print("\nLow temperature → peaked distribution (more deterministic)")
print("High temperature → flat distribution (more random)")

---

## 16. Key Formulas Reference

| Concept | Formula |
|---------|--------|
| Conditional probability | $P(A \mid B) = \frac{P(A \cap B)}{P(B)}$ |
| Bayes' theorem | $P(A \mid B) = \frac{P(B \mid A) P(A)}{P(B)}$ |
| Independence | $P(A \cap B) = P(A) P(B)$ |
| Binomial PMF | $P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$ |
| Poisson PMF | $P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}$ |
| Gaussian PDF | $\frac{1}{\sigma\sqrt{2\pi}} e^{-(x-\mu)^2 / 2\sigma^2}$ |
| Expectation (discrete) | $\mathbb{E}[X] = \sum_x x P(x)$ |
| Variance | $\text{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$ |
| Entropy | $H(X) = -\sum_x P(x) \log P(x)$ |
| Cross-entropy | $H(p,q) = -\sum_x p(x) \log q(x)$ |
| Naive Bayes | $P(C \mid \mathbf{x}) \propto P(C) \prod_i P(x_i \mid C)$ |

---

## 17. Summary

Probability provides the language for reasoning under uncertainty in artificial intelligence. **Conditional probability** and **Bayes' theorem** update beliefs from evidence. **Random variables** and **distributions** (Bernoulli, Binomial, Poisson, Gaussian) model real-world quantities. **Expectation** and **variance** summarize distributions. **Joint and conditional distributions** describe relationships between variables.

**Maximum likelihood estimation** connects probability to machine learning training. **Entropy** and **cross-entropy** measure uncertainty and serve as loss functions. **Naive Bayes** applies probability to classification. **Monte Carlo methods** approximate intractable computations through sampling.

Every probabilistic concept in this notebook appears in production AI: from the spam filter computing $P(\text{spam} \mid \text{words})$ to the language model computing $P(\text{next token} \mid \text{context})$. Probability is not a separate topic from AI — it is one of its core foundations.